# Week 3.1: Tool Use - Building AI Agents

## Key Insight: Thought → Action → Observation → Answer

Your digital twin learns to USE TOOLS autonomously!

## What You'll Build
- Tools your AI can call (calculator, search, etc.)
- Agent that decides when to use tools
- Multi-step reasoning loops
- Digital twin with tool capabilities

In [1]:
import json
from typing import Dict, List, Any, Callable
from datetime import datetime

print('Week 3.1: Tool Use & Agents')
print('Goal: Make your digital twin DO things, not just talk!')

Week 3.1: Tool Use & Agents
Goal: Make your digital twin DO things, not just talk!


## Part 1: Creating Tools

A tool is a function your AI can call.

In [7]:
print(calculator('25 * 4'))          # {'result': 100, ...}
print(calculator(' 47.50 * 0.15 '))  # {'result': 7.125, ...}
print(calculator('2 ** 10'))         # {'result': 1024, ...}
print(calculator('__import__("os")')) # {'error': 'Disallowed keyword...'}

{'result': 100, 'expression': '25 * 4'}
{'result': 7.125, 'expression': '47.50 * 0.15'}
{'result': 1024, 'expression': '2 ** 10'}
{'error': 'Invalid characters in: \'__import__("os")\''}


## Part 2: Tool Registry

Manage available tools.

In [9]:
TASK_STORE: List[Dict[str, Any]] = []


def add_task(description: str, due: str = None) -> Dict[str, Any]:
    """Add a task to the twin's task list."""
    if not isinstance(description, str) or not description.strip():
        return {'error': 'task description must be a non-empty string'}

    task = {
        'id': len(TASK_STORE) + 1,
        'description': description.strip(),
        'due': due,
        'done': False,
        'created_at': datetime.now().isoformat(timespec='seconds'),
    }
    TASK_STORE.append(task)
    return {'result': f'Added task #{task["id"]}: {task["description"]}', 'task': task}


def list_tasks(show_done: bool = False) -> Dict[str, Any]:
    """List tasks, optionally including completed ones."""
    tasks = TASK_STORE if show_done else [t for t in TASK_STORE if not t['done']]
    if not tasks:
        return {'result': 'No tasks', 'count': 0}
    lines = [f"#{t['id']} [{'x' if t['done'] else ' '}] {t['description']}"
             + (f" (due {t['due']})" if t['due'] else '')
             for t in tasks]
    return {'result': '\n'.join(lines), 'count': len(tasks)}


def complete_task(task_id: int) -> Dict[str, Any]:
    """Mark a task complete by ID."""
    for t in TASK_STORE:
        if t['id'] == task_id:
            t['done'] = True
            return {'result': f"Completed: {t['description']}"}
    return {'error': f'No task with id {task_id}'}

## Part 3: ReAct Agent

Agent that can reason and act with tools.

**ReAct Loop:**
1. **Thought**: Decide what to do
2. **Action**: Use a tool
3. **Observation**: See result
4. **Answer**: Respond to user (or loop again)

In [10]:
class SimpleReActAgent:
    """Agent with keyword-based routing (no LLM yet)."""

    def __init__(self, tool_registry):
        self.registry = tool_registry
        self.history = []

    def think(self, query: str) -> str:
        q = query.lower()

        # Memory: remember
        if any(k in q for k in ['remember', 'save this', 'note that', 'store']):
            return 'use_remember'
        # Memory: recall
        if any(k in q for k in ['recall', 'what do you know', 'what did i say',
                                'what is my', 'do you remember']):
            return 'use_recall'
        # Tasks: add
        if any(k in q for k in ['add task', 'remind me', 'todo', 'new task']):
            return 'use_add_task'
        # Tasks: list
        if any(k in q for k in ['list tasks', 'my tasks', 'show tasks', 'what tasks']):
            return 'use_list_tasks'
        # Tasks: complete
        if 'complete task' in q or 'done task' in q:
            return 'use_complete_task'
        # Calculator
        if any(w in q for w in ['calculate', 'what is', 'math',
                                '+', '-', '*', '/']) and any(c.isdigit() for c in q):
            return 'use_calculator'
        # Time
        if any(w in q for w in ['time', 'date', 'day']):
            return 'use_time'

        return 'direct_answer'

    def act(self, action: str, query: str) -> Dict[str, Any]:
        import re

        if action == 'use_calculator':
            # Extract math expression — no leading space, stop at '?'
            match = re.search(r'-?\d[\d\s+\-*/().]*', query)
            if match:
                expr = match.group(0).strip().rstrip('?')
                return self.registry.get_tool('calculator')(expr)
            return {'error': 'Could not extract expression'}

        if action == 'use_time':
            return self.registry.get_tool('get_time')()

        if action == 'use_remember':
            # Format: "remember that my favorite food is pasta"
            # or "remember favorite_food = pasta"
            m = re.search(
                r'remember(?:\s+that)?\s+(?:my\s+)?(.+?)\s+(?:is|=)\s+(.+)',
                query, re.IGNORECASE
            )
            if m:
                key = m.group(1).strip()
                value = m.group(2).strip().rstrip('.?!')
                return self.registry.get_tool('remember')(key, value)
            return {'error': 'Could not parse. Try: "remember my X is Y"'}

        if action == 'use_recall':
            # Try to extract a subject: "what is my X" or "recall X"
            m = re.search(r'(?:what is my|what\'s my|recall)\s+(.+)', query, re.IGNORECASE)
            key = m.group(1).strip().rstrip('.?!') if m else None
            return self.registry.get_tool('recall')(key)

        if action == 'use_add_task':
            # "remind me to call mom" or "add task: buy groceries"
            m = re.search(
                r'(?:remind me to|add task:?)\s+(.+)',
                query, re.IGNORECASE
            )
            if m:
                return self.registry.get_tool('add_task')(m.group(1).strip())
            return {'error': 'Try: "remind me to <task>"'}

        if action == 'use_list_tasks':
            return self.registry.get_tool('list_tasks')()

        if action == 'use_complete_task':
            m = re.search(r'(?:complete|done) task\s+(\d+)', query, re.IGNORECASE)
            if m:
                return self.registry.get_tool('complete_task')(int(m.group(1)))
            return {'error': 'Try: "complete task 3"'}

        return {'direct': query}

    def run(self, query: str) -> str:
        thought = self.think(query)
        self.history.append({'step': 'thought', 'content': thought})

        observation = self.act(thought, query)
        self.history.append({'step': 'action', 'content': observation})

        # Format the answer
        if isinstance(observation, dict):
            if 'result' in observation:
                return str(observation['result'])
            if 'error' in observation:
                return f"Error: {observation['error']}"

        return "I can help with calculations, time, memory, and tasks."

## Part 4: Integrating with Digital Twin

Add tools to your Week 1 digital twin.

In [5]:
class DigitalTwinWithTools:
    '''Digital twin that can use tools.'''

    def __init__(self, name: str):
        self.name = name
        self.registry = ToolRegistry()
        self.agent = None
        self.setup_tools()

    def setup_tools(self):
        '''Setup default tools.'''
        self.registry.register('calculator', calculator, 'Calculate math')
        self.registry.register('get_time', get_time, 'Get current time')
        self.agent = SimpleReActAgent(self.registry)

    def add_tool(self, name: str, func: Callable, description: str):
        '''Add a new tool.'''
        self.registry.register(name, func, description)

    def chat(self, message: str) -> str:
        '''Chat with tool use.'''
        return self.agent.run(message)

# Create digital twin with tools
twin = DigitalTwinWithTools('My Assistant')

print('\nDigital Twin with Tools:')
print(twin.chat('Calculate 47.50 * 0.15'))
print(twin.chat('What day is today?'))


Digital Twin with Tools:
The result is 7.125
It's 09:44:03 on 2026-10-06 (Tuesday)


## Exercise 1: Create Custom Tools

**Task**: Add 2 custom tools to your digital twin.

Ideas:
- Memory search
- Note taker
- Task list
- Habit tracker

In [11]:
registry = ToolRegistry()
registry.register('calculator',   calculator,   'Calculate math expressions')
registry.register('get_time',     get_time,     'Get current date and time')
registry.register('remember',     remember,     'Store a fact in memory')
registry.register('recall',       recall,       'Retrieve a stored fact')
registry.register('add_task',     add_task,     'Add a task to the list')
registry.register('list_tasks',   list_tasks,   'List current tasks')
registry.register('complete_task',complete_task,'Mark a task complete')

twin = DigitalTwinWithTools('Alex')
twin.registry = registry
twin.agent = SimpleReActAgent(registry)

print(registry.list_tools())
print()

# --- Test 1: Memory ---
print("Twin:", twin.chat("Remember that my favorite food is Italian pasta"))
print("Twin:", twin.chat("What is my favorite food?"))
print()

# --- Test 2: Tasks ---
print("Twin:", twin.chat("Remind me to call mom tomorrow"))
print("Twin:", twin.chat("Remind me to finish the RAG report"))
print("Twin:", twin.chat("List tasks"))
print()

# --- Test 3: Complete a task ---
print("Twin:", twin.chat("Complete task 1"))
print("Twin:", twin.chat("List tasks"))
print()

# --- Test 4: Calculator (now fixed) ---
print("Twin:", twin.chat("What is 25 * 4?"))
print("Twin:", twin.chat("Calculate 47.50 * 0.15"))
print()

# --- Test 5: Time ---
print("Twin:", twin.chat("What day is today?"))

Available tools:
- calculator: Calculate math expressions
- get_time: Get current date and time
- remember: Store a fact in memory
- recall: Retrieve a stored fact
- add_task: Add a task to the list
- list_tasks: List current tasks
- complete_task: Mark a task complete

Twin: Remembered: favorite_food = Italian pasta
Twin: Italian pasta

Twin: Added task #1: call mom tomorrow
Twin: Added task #2: finish the RAG report
Twin: #1 [ ] call mom tomorrow
#2 [ ] finish the RAG report

Twin: Completed: call mom tomorrow
Twin: #2 [ ] finish the RAG report

Twin: 100
Twin: 7.125

Twin: I can help with calculations, time, memory, and tasks.


## Summary

### What We Learned:
- Tools are functions AI can call
- Tool registry manages available tools
- ReAct pattern: Think → Act → Observe
- Agents can use tools autonomously
- Digital twin + tools = powerful assistant

### Next: Week 3.2 - Testing Your AI